## PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer


In [ ]:
FILE_PATH = "datasets/df_destx.csv"
SEP = ","
PCA_VARIANCE = 0.7
TEST_SIZE = 0.2
RANDOM_STATE = 42


In [ ]:
print("Starting Corrected PCA Pipeline...")
df = pd.read_csv(FILE_PATH, sep=SEP)
print(f"Raw CSV shape: {df.shape}")

# Detect label column automatically
label_col = None
for col in ["type", "class", "label", "phenotype", "target", "group"]:
    if col in df.columns:
        label_col = col
        break

if label_col is None:
    raise ValueError("No label column found in CSV. Please check your column names.")

y_series = df[label_col]
X_df = df.drop(columns=[label_col])


In [ ]:
# Force numeric and drop purely text columns
X_numeric = X_df.apply(pd.to_numeric, errors="coerce")
X_numeric = X_numeric.select_dtypes(include=[np.number]).dropna(axis=1, how="all")

# Transpose if genes are rows
if X_numeric.shape[0] > X_numeric.shape[1]:
    X_numeric = X_numeric.T
    print("Transposed data to Samples x Features")

print(f"Final Feature Shape: {X_numeric.shape}")


In [ ]:
# Impute missing values
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X_numeric)

# Log transform if data is non-negative
if np.min(X_imputed) >= 0:
    print("Applying Log2(x+1) transformation...")
    X_log = np.log2(X_imputed + 1)
else:
    X_log = X_imputed

#Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)


le = LabelEncoder()
y_encoded = le.fit_transform(y_series.astype(str))


In [ ]:

# Removed stratify to allow rare classes to process without errors
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"Training shape: {X_train.shape}")


In [ ]:

rf_base = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, rf_base.predict(X_test))
print(f"\n BASELINE")
print(f"Baseline Accuracy: {acc_base:.4f} (Using all {X_train.shape[1]} features)")


In [ ]:

print(f"\nPCA (Variance={PCA_VARIANCE})")
pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Features Compressed: {X_train.shape[1]} -> {X_train_pca.shape[1]} components")

rf_pca = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_pca.fit(X_train_pca, y_train)

acc_pca = accuracy_score(y_test, rf_pca.predict(X_test_pca))


In [ ]:
print(f"{'METRIC':<20} | {'ACCURACY':<10}")
print(f"{'Baseline (All Feats)':<20} | {acc_base:.4f}")
print(f"{'PCA Compressed':<20} | {acc_pca:.4f}")